In [1]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

In [2]:
df = pd.read_csv("../data/processed/featured.csv")
df.head()

,gender,married,dependents,education,self_employed,applicant_income,coapplicant_income,loan_amount,loan_term,credit_history,property_area,loan_status,total_income,emi_proxy,loan_to_income
0,Male,Yes,2,Not Graduate,No,6898.0,450.0,498.0,480.0,1.0,Urban,Y,7348.0,1.037498,0.067774
1,Male,Yes,0,Graduate,No,11532.0,3009.0,301.0,120.0,1.0,Semiurban,Y,14541.0,2.508312,0.020700
2,Female,Yes,3+,Graduate,No,2705.0,3419.0,67.0,480.0,1.0,Rural,Y,6124.0,0.139583,0.010941
3,Male,Yes,1,Graduate,No,13671.0,4394.0,453.0,360.0,NaN,Urban,Y,18065.0,1.258330,0.025076
4,Male,Yes,0,Graduate,No,8459.0,3433.0,477.0,360.0,NaN,Urban,N,11892.0,1.324996,0.040111


In [3]:
## Encode categorial data variables
label_gender=LabelEncoder()
df["gender"]=label_gender.fit_transform(df["gender"])


In [4]:
label_married=LabelEncoder()
df["married"]=label_married.fit_transform(df["married"])
label_education=LabelEncoder()
df["education"]=label_education.fit_transform(df["education"])
label_self_employed=LabelEncoder()
df["self_employed"]=label_self_employed.fit_transform(df["self_employed"])
label_loan_status=LabelEncoder()
df["loan_status"]=label_loan_status.fit_transform(df["loan_status"])


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   gender              1000 non-null   int64  
 1   married             1000 non-null   int64  
 2   dependents          1000 non-null   object 
 3   education           1000 non-null   int64  
 4   self_employed       1000 non-null   int64  
 5   applicant_income    1000 non-null   float64
 6   coapplicant_income  983 non-null    float64
 7   loan_amount         969 non-null    float64
 8   loan_term           1000 non-null   float64
 9   credit_history      912 non-null    float64
 10  property_area       1000 non-null   object 
 11  loan_status         1000 non-null   int64  
 12  total_income        983 non-null    float64
 13  emi_proxy           969 non-null    float64
 14  loan_to_income      952 non-null    float64
dtypes: float64(8), int64(5), object(2)
memory usage: 117.3+ 

In [6]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder
encoded_dependents=OneHotEncoder()
encoded1=encoded_dependents.fit_transform(df[["dependents"]])
encoded_geo_df1=pd.DataFrame(encoded1.toarray(),columns=encoded_dependents.get_feature_names_out(["dependents"]))
encoded_geo_df1.head()

,dependents_0,dependents_1,dependents_2,dependents_3+
0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0
4,1.0,0.0,0.0,0.0


In [7]:
# One hot encoding
encoded_property_area=OneHotEncoder()
encoded=encoded_property_area.fit_transform(df[["property_area"]])
encoded_geo_df=pd.DataFrame(encoded.toarray(),columns=encoded_property_area.get_feature_names_out(["property_area"]))
encoded_geo_df.head()

,property_area_Rural,property_area_Semiurban,property_area_Urban
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0


In [8]:
df=pd.concat([df,encoded_geo_df1,encoded_geo_df],axis=1)

df.head()

,gender,married,dependents,education,self_employed,applicant_income,coapplicant_income,loan_amount,loan_term,credit_history,...,total_income,emi_proxy,loan_to_income,dependents_0,dependents_1,dependents_2,dependents_3+,property_area_Rural,property_area_Semiurban,property_area_Urban
0,1,1,2,1,0,6898.0,450.0,498.0,480.0,1.0,...,7348.0,1.037498,0.067774,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1,1,0,0,0,11532.0,3009.0,301.0,120.0,1.0,...,14541.0,2.508312,0.020700,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,1,3+,0,0,2705.0,3419.0,67.0,480.0,1.0,...,6124.0,0.139583,0.010941,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,1,1,1,0,0,13671.0,4394.0,453.0,360.0,NaN,...,18065.0,1.258330,0.025076,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,1,1,0,0,0,8459.0,3433.0,477.0,360.0,NaN,...,11892.0,1.324996,0.040111,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [9]:
df.drop(columns=["dependents","property_area"],axis=1,inplace=True)

In [10]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   gender                   1000 non-null   int64  
 1   married                  1000 non-null   int64  
 2   education                1000 non-null   int64  
 3   self_employed            1000 non-null   int64  
 4   applicant_income         1000 non-null   float64
 5   coapplicant_income       983 non-null    float64
 6   loan_amount              969 non-null    float64
 7   loan_term                1000 non-null   float64
 8   credit_history           912 non-null    float64
 9   loan_status              1000 non-null   int64  
 10  total_income             983 non-null    float64
 11  emi_proxy                969 non-null    float64
 12  loan_to_income           952 non-null    float64
 13  dependents_0             1000 non-null   float64
 14  dependents_1             

In [11]:
X = df.drop('loan_status',axis=1)
y= df['loan_status']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=42,test_size=0.2)

In [13]:
X_train.describe()

,gender,married,education,self_employed,applicant_income,coapplicant_income,loan_amount,loan_term,credit_history,total_income,emi_proxy,loan_to_income,dependents_0,dependents_1,dependents_2,dependents_3+,property_area_Rural,property_area_Semiurban,property_area_Urban
count,800.000000,800.000000,800.000000,800.000000,800.000000,783.000000,776.000000,800.000000,730.000000,783.000000,776.000000,759.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000,800.000000
mean,0.813750,0.625000,0.221250,0.127500,8242.226250,3039.296296,277.162371,276.900000,0.857534,11276.704981,1.206998,0.029542,0.595000,0.167500,0.153750,0.083750,0.265000,0.331250,0.403750
std,0.389552,0.484426,0.415348,0.333741,3879.665955,1728.558698,130.645334,115.434164,0.349767,4239.614008,0.819676,0.022531,0.491199,0.373655,0.360935,0.277186,0.441609,0.470957,0.490955
min,0.000000,0.000000,0.000000,0.000000,1512.000000,8.000000,50.000000,120.000000,0.000000,2177.000000,0.122916,0.002668,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,5016.750000,1584.000000,159.750000,180.000000,1.000000,7973.500000,0.603122,0.014321,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,0.000000,0.000000,8222.500000,3070.000000,283.000000,240.000000,1.000000,11182.000000,1.009719,0.024518,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,0.000000,0.000000,11534.000000,4483.500000,388.250000,360.000000,1.000000,14480.500000,1.622907,0.037796,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,14999.000000,5994.000000,498.000000,480.000000,1.000000,20748.000000,4.141632,0.214254,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
model_type ="random_forest"
def _get_model():
    models = {
        "random_forest": RandomForestClassifier(
            n_estimators    = 200,
            max_depth       = 8,
            min_samples_leaf= 5,
            class_weight    = "balanced",
            random_state    = 42,
            n_jobs          = -1,
        ),
        "gradient_boost": GradientBoostingClassifier(
            n_estimators = 200,
            max_depth    = 8,
            random_state = 42,
        ),
        "logistic": LogisticRegression(
            max_iter     = 1000,
            class_weight = "balanced",
            random_state = 42,
        ),
    }
    model = models.get(model_type)
    if model is None:
        raise ValueError(f"Unknown model_type '{model_type}'. Choose from {list(models)}")
    return model

In [15]:
import sklearn
print(sklearn.__version__)

1.7.2
